# NOTEBOOK 06 — Modelos de Predicción por Checkpoints (v2.0 - Features Optimizadas)

## Objetivo
Construir y validar modelos predictivos de riesgo académico en dos momentos clave del semestre:
- **CP1 (Semana 6)**: Predicción temprana post-Parcial 1 - 11 semanas para intervención
- **CP2 (Semana 11)**: Predicción intermedia post-Parcial 2 - 6-7 semanas para intervención

## Cambios en v2.0
✅ **Features optimizadas** derivadas del análisis estadístico del Notebook 05
- CP1: 5 features + era_encoded (total 6)
- CP2: 12 features + era_encoded (total 13)

✅ **Visualizaciones mejoradas** del framework 06_framework_modelado_checkpoints

✅ **Documentación completa** de cada sección

---

# §0 — Setup, Imports y Configuración

In [ ]:
# Librerías estándar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from datetime import datetime

# Machine Learning
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, brier_score_loss, precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Configuración visual
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 7)

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

print("✅ Librerías cargadas correctamente")
print(f"📅 Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# §1 — Carga de Datos y Preparación

In [ ]:
# Cargar dataset completo
df_all = pd.read_csv('../data/dataset_completo_final.csv')

# Codificar era (antes de cualquier filtrado)
df_all['era_encoded'] = (df_all['era'] == 'formato_nuevo').astype(int)

# Crear target binario: 1 = REPROBÓ, 0 = APROBÓ
df_all['target_binario'] = (df_all['estado'] == 'REPROBÓ').astype(int)

# Subconjuntos clave para modelado
df_activos = df_all[df_all['estado'].isin(['APROBÓ', 'REPROBÓ'])].copy()
df_activos_engagement = df_activos[df_activos['engagement_hasta_p1'] > 0].copy()

# Información del dataset
print(f"Dataset completo: {df_all.shape}")
print(f"Estudiantes activos (sin retiro): {df_activos.shape}")
print(f"Activos con engagement: {df_activos_engagement.shape}")
print(f"\n📊 Distribución target binario:")
print(df_activos_engagement['target_binario'].value_counts())
print(f"\nTasa de reprobación: {df_activos_engagement['target_binario'].mean():.1%}")

# §2 — Definición de Features Optimizadas por Checkpoint

## CP1: 5 Features Principales + Control
Seleccionadas del análisis estadístico del Notebook 05 por máxima discriminación temprana:
- **parcial_1**: Evaluación sumativa principal (AUC: 0.81)
- **engagement_hasta_p1**: Horas en plataforma (AUC: 0.60)
- **parcial_1_modulos_unicos**: Diversidad de estudio (AUC: 0.60)
- **parcial_1_visitas_por_tema**: Profundidad de repaso (AUC: 0.57)
- **parcial_1_tiempo_por_visita**: Concentración en sesiones (AUC: 0.56)
- **era_encoded**: Control para diferencias estructurales

## CP2: 12 Features + Control
Agregan evaluaciones de P2 e indicadores de trayectoria:

In [ ]:
# ========== FEATURES OPTIMIZADAS CP1 ==========
FEATURES_CP1 = [
    'parcial_1',
    'engagement_hasta_p1',
    'parcial_1_modulos_unicos',
    'parcial_1_visitas_por_tema',
    'parcial_1_tiempo_por_visita',
    'era_encoded',
]

# ========== FEATURES OPTIMIZADAS CP2 ==========
FEATURES_CP2 = [
    'parcial_1', 'parcial_2', 'parcial_3',
    'promedio_ams', 'promedio_quices',
    'engagement_hasta_p1', 'parcial_1_modulos_unicos',
    'parcial_1_visitas', 'parcial_1_temas_unicos',
    'parcial_1_visitas_por_tema', 'parcial_1_tiempo_por_visita',
    'era_encoded',
]

print(f"✅ CP1: {len(FEATURES_CP1)} features")
print(f"   Features: {FEATURES_CP1}")
print(f"\n✅ CP2: {len(FEATURES_CP2)} features")
print(f"   Features: {FEATURES_CP2}")

# §3 — Preparación de Matrices de Features

### Estrategia de Manejo de Nulos
1. **Evaluaciones no presentadas**: Imputar con 0 (fuerte señal de riesgo)
2. **Engagement**: 0 si ausente (posible retiro early)
3. **Features por formato**: Crear indicadores binarios para features opcionales
4. **Trayectoria**: Derivar cambios P1→P2 para detectar mejora/declive

In [ ]:
def preparar_features_cp1(df):
    """Preparar matriz X para CP1 con imputación y feature engineering"""
    X = df[FEATURES_CP1].copy()
    
    # Imputar nulos a 0 (señal de riesgo)
    X = X.fillna(0)
    
    # Features derivadas para riesgo temprano
    X['no_presento_p1'] = (X['parcial_1'] == 0).astype(int)
    X['bajo_engagement_p1'] = (X['engagement_hasta_p1'] < X['engagement_hasta_p1'].quantile(0.25)).astype(int)
    X['bajo_modulos_p1'] = (X['parcial_1_modulos_unicos'] < 3).astype(int)
    
    return X

def preparar_features_cp2(df):
    """Preparar matriz X para CP2 incluyendo trayectoria P1→P2"""
    X = df[FEATURES_CP2].copy()
    
    # Imputar nulos a 0
    X = X.fillna(0)
    
    # Features derivadas
    X['no_presento_p1'] = (X['parcial_1'] == 0).astype(int)
    X['no_presento_p2'] = (X['parcial_2'] == 0).astype(int)
    X['no_presento_p3'] = (X['parcial_3'] == 0).astype(int)
    
    # Trayectoria
    X['mejora_p2_p1'] = (X['parcial_2'] - X['parcial_1']).clip(lower=-1, upper=1)
    X['mejora_p3_p2'] = (X['parcial_3'] - X['parcial_2']).clip(lower=-1, upper=1)
    
    # Recuperación
    X['se_recupero_p2'] = ((X['parcial_1'] < 3) & (X['parcial_2'] >= 3)).astype(int)
    
    # Consistencia académica
    X['std_parciales'] = df[['parcial_1', 'parcial_2', 'parcial_3']].std(axis=1).fillna(0)
    
    return X

# Aplicar a dataset de entrenamiento
X_cp1 = preparar_features_cp1(df_activos_engagement)
y_cp1 = df_activos_engagement['target_binario']

X_cp2 = preparar_features_cp2(df_activos_engagement)
y_cp2 = df_activos_engagement['target_binario']

print(f"✅ CP1: {X_cp1.shape}")
print(f"   Columnas finales: {list(X_cp1.columns)}")
print(f"\n✅ CP2: {X_cp2.shape}")
print(f"   Columnas finales: {list(X_cp2.columns)}")

# §4 — Estrategia de Validación Temporal

### Walk-Forward Validation
Respeta la estructura temporal del curso (5 semestres):
- **Split 1**: Train=[202320], Val=[202410], Test=[202420]
- **Split 2**: Train=[202320,202410], Val=[202420], Test=[202510]
- **Split 3**: Train=[202320,202410,202420], Val=[202510], Test=[202520]

In [ ]:
# Semestres en orden cronológico
semestres = sorted(df_all['semestre'].unique())
print(f"Semestres: {semestres}")

def crear_splits_temporales(df, features_func, features_list):
    """
    Crear splits temporales respetando la estructura de cohortes.
    
    Args:
        df: DataFrame completo
        features_func: función para preparar features
        features_list: lista de features a usar
    
    Returns:
        Lista de tuplas (X_train, y_train, X_val, y_val, X_test, y_test)
    """
    splits = []
    
    for i in range(len(semestres) - 2):
        train_sems = semestres[:i+1]
        val_sem = semestres[i+1]
        test_sem = semestres[i+2]
        
        # Subsets
        df_train = df[df['semestre'].isin(train_sems)]
        df_val = df[df['semestre'] == val_sem]
        df_test = df[df['semestre'] == test_sem]
        
        # Features
        X_train = features_func(df_train)[features_list]
        y_train = df_train['target_binario']
        
        X_val = features_func(df_val)[features_list]
        y_val = df_val['target_binario']
        
        X_test = features_func(df_test)[features_list]
        y_test = df_test['target_binario']
        
        splits.append({
            'train': (X_train, y_train, train_sems),
            'val': (X_val, y_val, val_sem),
            'test': (X_test, y_test, test_sem)
        })
        
        print(f"Split {i+1}: Train={train_sems} ({len(df_train)}) → "
              f"Val={val_sem} ({len(df_val)}) → Test={test_sem} ({len(df_test)})")
    
    return splits

# Crear splits
splits_cp1 = crear_splits_temporales(df_activos_engagement, preparar_features_cp1, FEATURES_CP1)
splits_cp2 = crear_splits_temporales(df_activos_engagement, preparar_features_cp2, FEATURES_CP2)

# §5 — Funciones de Evaluación y Visualización

In [ ]:
def evaluar_modelo(y_true, y_pred, y_proba):
    """
    Calcular métricas de evaluación para clasificación binaria.
    
    Returns:
        dict con métricas: AUC, F1, Recall, Precision, Brier
    """
    return {
        'AUC': roc_auc_score(y_true, y_proba),
        'F1': f1_score(y_true, y_pred, average='macro'),
        'Recall': recall_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Brier': brier_score_loss(y_true, y_proba),
        'Accuracy': accuracy_score(y_true, y_pred)
    }

def graficar_resultados_comparativos(resultados_df, titulo_cp):
    """
    Visualización mejorada de desempeño de modelos.
    Combina estética del framework con claridad de 06_modelos.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Panel 1: Métricas clave
    metricas = ['AUC', 'F1', 'Recall', 'Precision']
    df_plot = resultados_df.groupby('Modelo')[metricas].mean()
    
    df_plot.plot(kind='bar', ax=axes[0], width=0.8, edgecolor='black', linewidth=0.5)
    axes[0].set_title(f'{titulo_cp}: Comparación de Métricas', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Puntaje', fontsize=11)
    axes[0].axhline(0.75, color='red', linestyle='--', alpha=0.5, label='Target (0.75)')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Panel 2: Evolución por split
    for modelo in resultados_df['Modelo'].unique():
        datos = resultados_df[resultados_df['Modelo'] == modelo]
        axes[1].plot(datos['Split'], datos['AUC'], marker='o', label=modelo, linewidth=2)
    
    axes[1].set_title(f'{titulo_cp}: Estabilidad AUC por Período', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Split Temporal', fontsize=11)
    axes[1].set_ylabel('AUC', fontsize=11)
    axes[1].axhline(0.75, color='red', linestyle='--', alpha=0.5)
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def graficar_matriz_confusion_dashboard(y_true, y_pred, titulo):
    """
    Dashboard de errores: matriz de confusión con anotaciones claras.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Aprobará', 'Reprobará'])
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    
    ax.set_title(titulo, fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    return cm

print("✅ Funciones de evaluación cargadas")

# §6 — Entrenamiento de Modelos: CP1

## Punto de Control 1 (Semana 6, Post-Parcial 1)
**Objetivo**: Predicción temprana para activar intervenciones en semana 6
**Ventana de acción**: 11 semanas hasta fin de semestre

In [ ]:
def entrenar_modelos_cp1(splits):
    """
    Entrenar 4 baselines para CP1 con validación temporal.
    
    Modelos:
    1. Baseline 0: Predictor mayoritario (piso)
    2. Baseline 1: Regresión logística univariada (solo P1)
    3. Baseline 2: Regresión logística completa
    4. Baseline 3: Árbol de decisión interpretable
    """
    resultados = []
    
    for split_idx, split in enumerate(splits):
        X_train, y_train = split['train'][0], split['train'][1]
        X_test, y_test = split['test'][0], split['test'][1]
        
        print(f"\n--- Split {split_idx + 1} ---")
        print(f"Train: {X_train.shape} | Test: {X_test.shape}")
        
        # 1. Baseline 0: Mayoritario
        y_pred = np.full(len(y_test), y_train.mode()[0])
        y_proba = np.where(y_pred == 1, 0.5, 0.5)
        metricas = evaluar_modelo(y_test, y_pred, y_proba)
        resultados.append({'Split': split_idx, 'Modelo': 'Baseline 0 (Mayoritario)', **metricas})
        
        # 2. Baseline 1: Univariado (solo parcial_1)
        model = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))])
        model.fit(X_train[['parcial_1']], y_train)
        y_pred = model.predict(X_test[['parcial_1']])
        y_proba = model.predict_proba(X_test[['parcial_1']])[:, 1]
        metricas = evaluar_modelo(y_test, y_pred, y_proba)
        resultados.append({'Split': split_idx, 'Modelo': 'Baseline 1 (LR univariada)', **metricas})
        
        # 3. Baseline 2: Regresión logística con todas las features
        model = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))])
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        metricas = evaluar_modelo(y_test, y_pred, y_proba)
        resultados.append({'Split': split_idx, 'Modelo': 'Baseline 2 (LR completa)', **metricas})
        
        # 4. Baseline 3: Árbol de decisión
        model = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=SEED)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        metricas = evaluar_modelo(y_test, y_pred, y_proba)
        resultados.append({'Split': split_idx, 'Modelo': 'Baseline 3 (Árbol)', **metricas})
        
        print("✅ Split completado")
    
    return pd.DataFrame(resultados)

# Entrenar CP1
print("🚀 Entrenando modelos CP1...")
resultados_cp1 = entrenar_modelos_cp1(splits_cp1)
print("\n✅ CP1 completado")

# §7 — Entrenamiento de Modelos: CP2

In [ ]:
def entrenar_modelos_cp2(splits):
    """
    Entrenar 4 baselines para CP2 con validación temporal.
    Similar a CP1 pero con features expandidas.
    """
    resultados = []
    
    for split_idx, split in enumerate(splits):
        X_train, y_train = split['train'][0], split['train'][1]
        X_test, y_test = split['test'][0], split['test'][1]
        
        print(f"\n--- Split {split_idx + 1} ---")
        print(f"Train: {X_train.shape} | Test: {X_test.shape}")
        
        # 1. Baseline 0: Mayoritario
        y_pred = np.full(len(y_test), y_train.mode()[0])
        y_proba = np.where(y_pred == 1, 0.5, 0.5)
        metricas = evaluar_modelo(y_test, y_pred, y_proba)
        resultados.append({'Split': split_idx, 'Modelo': 'Baseline 0 (Mayoritario)', **metricas})
        
        # 2. Baseline 1: Univariado (solo parcial_2)
        model = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))])
        model.fit(X_train[['parcial_2']], y_train)
        y_pred = model.predict(X_test[['parcial_2']])
        y_proba = model.predict_proba(X_test[['parcial_2']])[:, 1]
        metricas = evaluar_modelo(y_test, y_pred, y_proba)
        resultados.append({'Split': split_idx, 'Modelo': 'Baseline 1 (LR univariada P2)', **metricas})
        
        # 3. Baseline 2: Regresión logística con todas las features
        model = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))])
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        metricas = evaluar_modelo(y_test, y_pred, y_proba)
        resultados.append({'Split': split_idx, 'Modelo': 'Baseline 2 (LR completa)', **metricas})
        
        # 4. Baseline 3: Árbol de decisión
        model = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=SEED)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        metricas = evaluar_modelo(y_test, y_pred, y_proba)
        resultados.append({'Split': split_idx, 'Modelo': 'Baseline 3 (Árbol)', **metricas})
        
        print("✅ Split completado")
    
    return pd.DataFrame(resultados)

# Entrenar CP2
print("🚀 Entrenando modelos CP2...")
resultados_cp2 = entrenar_modelos_cp2(splits_cp2)
print("\n✅ CP2 completado")

# §8 — Análisis de Resultados CP1

In [ ]:
print("\n" + "="*70)
print("RESUMEN DE DESEMPEÑO — CHECKPOINT 1 (Semana 6)")
print("="*70)

resumen_cp1 = resultados_cp1.groupby('Modelo')[['AUC', 'F1', 'Recall', 'Precision']].mean()
print("\n📊 Desempeño Promedio por Modelo:")
print(resumen_cp1.round(3))

print("\n📈 Interpretación:")
for modelo in resumen_cp1.index:
    auc = resumen_cp1.loc[modelo, 'AUC']
    recall = resumen_cp1.loc[modelo, 'Recall']
    
    if auc > 0.80:
        status = "⭐ EXCELENTE"
    elif auc > 0.70:
        status = "✅ BUENO"
    else:
        status = "⚠️ LIMITADO"
    
    print(f"  {modelo}: AUC={auc:.3f}, Recall={recall:.2f} {status}")

graficar_resultados_comparativos(resultados_cp1, "CP1 (Semana 6)")

# §9 — Análisis de Resultados CP2

In [ ]:
print("\n" + "="*70)
print("RESUMEN DE DESEMPEÑO — CHECKPOINT 2 (Semana 11)")
print("="*70)

resumen_cp2 = resultados_cp2.groupby('Modelo')[['AUC', 'F1', 'Recall', 'Precision']].mean()
print("\n📊 Desempeño Promedio por Modelo:")
print(resumen_cp2.round(3))

print("\n📈 Interpretación:")
for modelo in resumen_cp2.index:
    auc = resumen_cp2.loc[modelo, 'AUC']
    recall = resumen_cp2.loc[modelo, 'Recall']
    
    if auc > 0.80:
        status = "⭐ EXCELENTE"
    elif auc > 0.70:
        status = "✅ BUENO"
    else:
        status = "⚠️ LIMITADO"
    
    print(f"  {modelo}: AUC={auc:.3f}, Recall={recall:.2f} {status}")

graficar_resultados_comparativos(resultados_cp2, "CP2 (Semana 11)")

# §10 — Análisis de Matrices de Confusión

### Interpretación de Errores
- **Falsos Positivos (FP)**: Estudiante predicho como riesgo pero aprobó
  - Genera "fatiga de alertas" en tutores
  - Mejor que Falsos Negativos pero costoso

- **Falsos Negativos (FN)**: Estudiante predicho como seguro pero reprobó
  - **CRÍTICO**: Oportunidad de intervención perdida
  - Impacto directo en estudiantes

**Métrica clave para CP1/CP2**: Maximizar RECALL (minimizar FN)

In [ ]:
# Matriz de confusión para mejor modelo de cada CP
# (Usamos el último split como representativo)

best_model_cp1 = resultados_cp1.groupby('Modelo')['AUC'].mean().idxmax()
best_model_cp2 = resultados_cp2.groupby('Modelo')['AUC'].mean().idxmax()

print(f"🏆 Mejor modelo CP1: {best_model_cp1}")
print(f"🏆 Mejor modelo CP2: {best_model_cp2}")

# Nota: Para visualización completa, entrenar en el último split
X_train_cp1, y_train_cp1 = splits_cp1[-1]['train'][0], splits_cp1[-1]['train'][1]
X_test_cp1, y_test_cp1 = splits_cp1[-1]['test'][0], splits_cp1[-1]['test'][1]

X_train_cp2, y_train_cp2 = splits_cp2[-1]['train'][0], splits_cp2[-1]['train'][1]
X_test_cp2, y_test_cp2 = splits_cp2[-1]['test'][0], splits_cp2[-1]['test'][1]

# Entrenar best models
if best_model_cp1 == 'Baseline 2 (LR completa)':
    model_cp1 = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))])
    model_cp1.fit(X_train_cp1, y_train_cp1)
    y_pred_cp1 = model_cp1.predict(X_test_cp1)
elif best_model_cp1 == 'Baseline 3 (Árbol)':
    model_cp1 = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=SEED)
    model_cp1.fit(X_train_cp1, y_train_cp1)
    y_pred_cp1 = model_cp1.predict(X_test_cp1)

if best_model_cp2 == 'Baseline 2 (LR completa)':
    model_cp2 = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))])
    model_cp2.fit(X_train_cp2, y_train_cp2)
    y_pred_cp2 = model_cp2.predict(X_test_cp2)
elif best_model_cp2 == 'Baseline 3 (Árbol)':
    model_cp2 = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=SEED)
    model_cp2.fit(X_train_cp2, y_train_cp2)
    y_pred_cp2 = model_cp2.predict(X_test_cp2)

print("\n🔍 Análisis de Errores:")
graficar_matriz_confusion_dashboard(y_test_cp1, y_pred_cp1, f"Matriz de Confusión - CP1 | {best_model_cp1}")
graficar_matriz_confusion_dashboard(y_test_cp2, y_pred_cp2, f"Matriz de Confusión - CP2 | {best_model_cp2}")

# §11 — Conclusiones y Recomendaciones

## Resumen Ejecutivo

### Checkpoint 1 (Semana 6)
- **Objetivo**: Predicción temprana con máxima ventana de intervención
- **Features optimizadas**: 5 indicadores de rendimiento+engagement (6 con era)
- **Desempeño esperado**: AUC ~0.75-0.79 (BUENO)
- **Uso**: Alertas tempranas para tutorías de refuerzo

### Checkpoint 2 (Semana 11)
- **Objetivo**: Confirmación de riesgo con dos parciales de evidencia
- **Features optimizadas**: 12 indicadores académicos+engagement (13 con era)
- **Desempeño esperado**: AUC ~0.80-0.90+ (EXCELENTE)
- **Uso**: Alertas dirigidas para intervenciones intensivas finales

## Próximos Pasos
1. ✅ Validar modelos en semestres futuros (202620+)
2. ⏳ Implementar sistema de alertas en Bloque Neón
3. ⏳ Calibración de umbrales según niveles de riesgo (CRÍTICO, ALTO, MODERADO, SEGUIMIENTO)
4. ⏳ Evaluación de efectividad de intervenciones basadas en predicciones